# MODUL PRAKTIKUM BIG DATA
## Pertemuan 4 — Instalasi Apache Spark & Pengenalan PySpark DataFrame (Arch Linux Safe Edition)

| Informasi | Keterangan |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **Dosen Pengampu** | Alifia Revan Prananda, S.Pd., M.Eng |
| **Sistem Operasi** | Arch Linux / CachyOS (Lingkungan Terisolasi & Aman) |
| **Direktori Kerja** | `/mnt/alterra/repos/praktikum-big-data` (uv project & Git repo) |
| **Perangkat Lunak** | Apache Spark 3.5.9, PySpark 3.5.9, Apache Hadoop 3.4.3, OpenJDK 11 |
| **Topik Pembahasan** | Konsep Spark vs MapReduce, Mode Local (`local[*]`), PySpark DataFrame API, Integrasi HDFS, Studi Kasus E-Commerce |

---

### 🛡️ Jaminan Keamanan Sistem (Zero-Conflict Arch Linux)
Modul versi ini disesuaikan khusus untuk lingkungan **Arch Linux / CachyOS** Anda agar tetap bersih, cepat, dan terisolasi tanpa risiko bentrok dependensi sistem:
1. **Java 11 Terisolasi Penuh**: Spark 3.5.9 berjalan di atas Java 11 (`jdk11-openjdk`) yang dikunci lewat konfigurasi internal `spark-env.sh` dan inisialisasi sesi kernel. Default Java 21 sistem Anda tetap aman dan tidak terganggu.
2. **Lingkungan Portabel (Userspace `~/spark`)**: Seluruh biner Spark ditempatkan di folder `~/spark` milik pengguna. Sama sekali **tidak memerlukan akses root (`sudo`)** untuk operasi rutin.
3. **Integrasi Fish Shell Modern**: Lokasi biner Spark didaftarkan via `fish_add_path ~/spark/bin ~/spark/sbin`, sehingga seluruh perintah CLI (`spark-submit`, dll.) dapat dipanggil langsung dari terminal maupun notebook tanpa mengotori `.bashrc`.
4. **Manajemen Dependensi via `uv`**: Paket `pyspark==3.5.9` dikelola langsung di dalam virtual environment proyek menggunakan `uv add pyspark==3.5.9`, memastikan isolasi total dari paket Python sistem.
5. **Standar Variabel Lingkungan (`$USER`)**: Sesuai standar Unix, seluruh jalur direktori HDFS menggunakan variabel dinamis `$USER` (otomatis memetakan username aktif Anda, misal `/user/kaky/...`).

---
## 1. Landasan Teori & Arsitektur Apache Spark

### Recap Pertemuan Sebelumnya
Sebelum memulai, pastikan daemon Hadoop dari Pertemuan 3 sudah berjalan:
- Jalankan perintah terminal: `sudo systemctl start sshd && start-dfs.sh && start-yarn.sh`
- Verifikasi dengan `jps`: pastikan terdapat minimal 5 daemon aktif (`NameNode`, `DataNode`, `SecondaryNameNode`, `ResourceManager`, `NodeManager`).

---

### 1.1 Mengapa Kita Butuh Apache Spark?
Pada Pertemuan 3, kita mempelajari **Hadoop MapReduce**. Meskipun andal untuk pemrosesan terdistribusi berskala masif, MapReduce memiliki satu kelemahan besar: **setiap tahap perantara (intermediate state) wajib ditulis dan dibaca kembali dari disk fisik (HDFS)**. Hal ini menimbulkan *I/O bottleneck* yang parah pada pemrosesan berulang (iteratif) dan analisis data interaktif.

**Apache Spark** hadir sebagai terobosan komputasi Big Data generasi baru:
* **In-Memory Computing**: Spark memproses dan mempertahankan data di dalam **memori RAM**, menghasilkan kecepatan **10 hingga 100 kali lebih cepat** dibandingkan MapReduce.
* **Integrasi Ekosistem**: Spark tidak menggantikan Hadoop, melainkan melengkapinya. Spark bertindak sebagai lapisan komputasi super cepat yang membaca dan menulis data langsung dari dan ke **HDFS** yang sudah kita bangun.

---

### 1.2 Komponen Arsitektur Apache Spark

| Komponen | Peran dan Tanggung Jawab |
|---|---|
| **Driver Program** | Proses utama yang menjalankan fungsi `main()` aplikasi Spark kita, menginisialisasi `SparkSession`, serta menerjemahkan kode menjadi rencana eksekusi (DAG). |
| **Cluster Manager** | Mengalokasikan sumber daya komputasi. Dapat berupa mode `local`, Standalone, Mesos, Kubernetes, atau **Hadoop YARN** yang telah kita pasang di Pertemuan 3. |
| **Worker / Executor** | Proses pekerja yang bertugas mengeksekusi unit komputasi (*task*) dan menyimpan blok data di dalam memori/RAM. |

---

### 1.3 RDD vs DataFrame: Mengapa Memilih DataFrame?
* **RDD (*Resilient Distributed Dataset*)**: Abstraksi data dasar di Spark — kumpulan objek Java/Python terdistribusi tanpa skema kolom eksplisit. Fleksibel, namun minim optimasi otomatis.
* **DataFrame**: Abstraksi tingkat tinggi berbentuk tabel dengan baris dan kolom bertipe data (mirip tabel SQL atau Pandas DataFrame).
  - **Catalyst Optimizer**: Mesin optimasi kueri otomatis yang menyusun rencana eksekusi paling efisien sebelum program dijalankan.
  - **Tungsten Execution Engine**: Pengelola alokasi memori tingkat rendah (*off-heap memory*) yang memaksimalkan efisiensi CPU dan RAM.

> 💡 **Mode Praktikum Ini:** Kita menjalankan Spark dalam mode **`local[*]`** — seluruh proses *driver* dan *executor* memanfaatkan seluruh *core thread* CPU laptop Anda secara maksimal tanpa *overhead* jaringan klaster.

---
## 2. Panduan Lengkap Instalasi Apache Spark di Arch Linux (Terminal)

Ikuti langkah-langkah di bawah ini di terminal Anda untuk memasang Apache Spark 3.5.9 dan mengintegrasikannya dengan lingkungan Arch Linux Anda.

---

### Langkah A: Mengunduh dan Mengekstrak Apache Spark 3.5.9
Buka terminal dan jalankan:

```bash
cd ~
wget https://downloads.apache.org/spark/spark-3.5.9/spark-3.5.9-bin-hadoop3.tgz
tar -xzvf spark-3.5.9-bin-hadoop3.tgz
mv spark-3.5.9-bin-hadoop3 spark
rm spark-3.5.9-bin-hadoop3.tgz
```

> **Catatan:** Varian `bin-hadoop3` dibangun khusus agar kompatibel penuh dengan Hadoop 3.4.3 yang telah terpasang di laptop Anda.

---

### Langkah B: Kunci Java 11 Khusus Spark di `spark-env.sh` (Kunci Isolasi)
Agar Spark selalu memanggil OpenJDK 11 (bukan Java 21 bawaan Arch Linux), buat berkas konfigurasi `spark-env.sh`:

```bash
cp ~/spark/conf/spark-env.sh.template ~/spark/conf/spark-env.sh
echo "export JAVA_HOME=/usr/lib/jvm/java-11-openjdk" >> ~/spark/conf/spark-env.sh
echo "export SPARK_LOCAL_IP=127.0.0.1" >> ~/spark/conf/spark-env.sh
```

---

### Langkah C: Mendaftarkan Spark ke PATH (Khusus Fish Shell)
Daftarkan biner Spark agar dapat dipanggil dari seluruh sesi terminal dan notebook:

```bash
fish_add_path ~/spark/bin ~/spark/sbin
```

Verifikasi bahwa biner Spark dapat dipanggil:
```bash
spark-submit --version
```
*(Akan muncul banner ASCII Apache Spark beserta tulisan `version 3.5.9`).*

---

### Langkah D: Menginstal Paket `pyspark` di Repositori Proyek via `uv`
Masuk ke repositori praktikum kita dan pasang paket PySpark ke virtual environment proyek:

```bash
cd /mnt/alterra/repos/praktikum-big-data
uv add pyspark==3.5.9
```

> ⚠️ **Penting:** Versi paket Python `pyspark==3.5.9` wajib sama persis dengan versi biner standalone Spark di `~/spark` untuk mencegah ketidakcocokan protokol RPC.

---
## 3. Melanjutkan di Jupyter Notebook: Hands-On PySpark

Jalankan Jupyter Notebook dari dalam folder proyek menggunakan `uv`:

```bash
cd /mnt/alterra/repos/praktikum-big-data
uv run jupyter notebook
```

---

### 3.1 Menginisialisasi `SparkSession`
`SparkSession` adalah gerbang utama (*entry point*) untuk seluruh interaksi koding dengan PySpark DataFrame. Di balik layar, `SparkSession` membuat koneksi ke *driver*, mengatur alokasi memori, serta menginisialisasi konteks Spark.

In [ ]:
import os
import sys
import getpass
from pyspark.sql import SparkSession

# 1. Kunci JAVA_HOME ke OpenJDK 11 untuk sub-shell kernel ini
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 2. Inisialisasi SparkSession dalam mode local[*]
spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark-ArchLinux") \
    .master("local[*]") \
    .getOrCreate()

# 3. Mereduksi pesan log diagnostik agar output notebook rapi
spark.sparkContext.setLogLevel("ERROR")

USERNAME = getpass.getuser()
print("=== STATUS KONEKSI SPARK SESSION ===")
print(f"User Aktif   : {USERNAME}")
print(f"Versi Spark  : {spark.version}")
print(f"Master Mode  : {spark.sparkContext.master}")
print(f"Aplikasi     : {spark.sparkContext.appName}")
print("Status       : SparkSession aktif & terisolasi penuh!")

Jika output menampilkan `Versi Spark: 3.5.9` dan `Status: SparkSession aktif`, lingkungan PySpark di laptop Arch Linux Anda telah berhasil terhubung sempurna!

---

## 4. Memuat Data ke Spark DataFrame

Kita akan menggunakan kembali skenario dataset transaksi e-commerce (serupa dengan Pertemuan 2-3) untuk mempraktikkan operasi manipulasi data terdistribusi.

In [ ]:
# Membuat ulang dataset transaksi lokal jika belum ada di direktori kerja
import os

if not os.path.exists("data_transaksi_ecommerce.csv"):
    import numpy as np
    import pandas as pd
    
    np.random.seed(42)
    n = 600
    kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
    kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
    metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
    tanggal_range = pd.date_range("2026-07-01", "2026-07-31", freq="D")
    
    data = {
        "order_id": [f"ORD-{1000+i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "kota": np.random.choice(kota_list, size=n),
        "unit_terjual": np.random.randint(1, 10, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000, 500000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n, p=[0.35, 0.30, 0.20, 0.15]),
    }
    pd.DataFrame(data).to_csv("data_transaksi_ecommerce.csv", index=False)
    print("Dataset 'data_transaksi_ecommerce.csv' berhasil dibuat (600 baris).")
else:
    print("Dataset 'data_transaksi_ecommerce.csv' sudah tersedia.")

In [ ]:
# Membaca berkas CSV lokal ke dalam Spark DataFrame
# header=True      -> Menganggap baris pertama sebagai header kolom
# inferSchema=True  -> Spark otomatis mendeteksi tipe data kolom (integer, string, double)
df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True)

print("Tipe Objek Python:", type(df))
print("
--- SKEMA STRUKTUR DATA (printSchema) ---")
df.printSchema()

Perhatikan bahwa metode `printSchema()` menampilkan nama kolom, tipe data, serta penanda *nullable* (apakah kolom boleh bernilai null). Ini sangat penting pada Big Data di mana dataset berukuran terlalu besar untuk diinspeksi secara manual.

In [ ]:
# Menampilkan 5 baris pertama (setara dengan df.head() pada Pandas)
df.show(5)

# Menghitung total jumlah baris data (setara dengan len(df) pada Pandas)
print("Total Jumlah Baris Transaksi:", df.count())

### 4.1 Bridging: Perbandingan Sintaks Pandas vs PySpark

Bagi Anda yang sudah terbiasa dengan Pandas di Pertemuan 2, tabel berikut merangkum kesetaraan operasinya di PySpark:

| Operasi Analisis | Pandas | PySpark DataFrame |
|---|---|---|
| Menampilkan data teratas | `df.head(5)` | `df.show(5)` |
| Menghitung jumlah baris | `len(df)` | `df.count()` |
| Memeriksa skema & tipe | `df.info()` | `df.printSchema()` |
| Memilih subset kolom | `df[["kolom1", "kolom2"]]` | `df.select("kolom1", "kolom2")` |
| Menyaring baris data | `df[df["kolom"] > 5]` | `df.filter(df["kolom"] > 5)` |
| Menambah kolom kalkulasi | `df["c"] = df["a"] * df["b"]` | `df.withColumn("c", col("a") * col("b"))` |
| Mengelompokkan data | `df.groupby("k").sum()` | `df.groupBy("k").sum()` / `agg(...)` |

---

### ⚠️ Konsep Fundamental: *Lazy Evaluation* vs *Eager Execution*
* **Pandas (*Eager Execution*)**: Setiap baris kode yang dieksekusi langsung memproses data di memori RAM dan mengembalikan hasil seketika.
* **PySpark (*Lazy Evaluation*)**: Operasi transformasi (seperti `select()`, `filter()`, `withColumn()`) **tidak langsung memproses data secara fisik**. Spark hanya mencatat rencana logika (*DAG - Directed Acyclic Graph*). Perhitungan komputasi sesungguhnya baru dieksekusi ketika ada perintah **Aksi (*Action*)** yang dipanggil, seperti `.show()`, `.count()`, atau `.write`.
* **Keuntungan**: *Catalyst Optimizer* dapat menganalisis seluruh alur kueri Anda terlebih dahulu, membuang kolom yang tidak perlu (*column pruning*), dan menerapkan filter sedini mungkin (*predicate pushdown*) sebelum data dibaca dari disk.

---
## 5. Operasi Dasar DataFrame: Select, Filter, GroupBy, & Agg

### 5.1 `select()` — Memilih Kolom Tertentu
Digunakan untuk memproyeksikan kolom-kolom tertentu yang relevan untuk analisis.

In [ ]:
# Memilih subset kolom: order_id, kota, dan kategori
df.select("order_id", "kota", "kategori").show(5)

### 5.2 `filter()` — Menyaring Baris Data
Kita dapat menggunakan fungsi `col("nama_kolom")` dari modul `pyspark.sql.functions` untuk menuliskan ekspresi kondisi logis.

In [ ]:
from pyspark.sql.functions import col

# Menyaring transaksi kategori 'Elektronik' dengan unit_terjual > 5
df_filtered = df.filter((col("kategori") == "Elektronik") & (col("unit_terjual") > 5))
print("Jumlah transaksi Elektronik dengan unit > 5:", df_filtered.count())
df_filtered.show(5)

### 5.3 `withColumn()`, `groupBy()`, dan `agg()` — Agregasi Data Lanjutan
* **`withColumn()`**: Menambahkan atau memperbarui kolom data.
* **`groupBy()` + `agg()`**: Mengelompokkan baris dan menjalankan fungsi ringkasan seperti `sum()`, `avg()`, dan `count()`.

In [ ]:
from pyspark.sql.functions import sum as spark_sum, count, avg

# 1. Tambahkan kolom baru: total_pendapatan = unit_terjual * harga_satuan
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# 2. Meringkas performa penjualan per kota
ringkasan_kota = df.groupBy("kota").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan"),
    count("order_id").alias("jumlah_transaksi"),
    avg("unit_terjual").alias("rata_rata_unit")
).orderBy(col("total_pendapatan").desc())

print("=== RINGKASAN PENJUALAN PER KOTA ===")
ringkasan_kota.show()

> 💡 **Tips Praktik Terbaik:** Perhatikan bahwa kita menggunakan `from pyspark.sql.functions import sum as spark_sum`. Pemberian alias `spark_sum` ini sangat dianjurkan untuk mencegah bentrok nama dengan fungsi `sum()` bawaan Python standar.

---
## 6. Integrasi Apache Spark + HDFS (Kekuatan Utama Big Data)

Karena klaster HDFS lokal Anda sudah berjalan sejak Pertemuan 3, Spark dapat membaca data **langsung dari HDFS secara native** menggunakan skema URL `hdfs://localhost:9000/...` tanpa perlu menyalin atau mengunduh data ke penyimpanan lokal laptop terlebih dahulu!

In [ ]:
# 1. Menyiapkan folder direktori praktikum di HDFS menggunakan variabel dinamis $USER
!hdfs dfs -mkdir -p /user/$USER/pertemuan4

# 2. Mengunggah dataset ke direktori HDFS
!hdfs dfs -put -f data_transaksi_ecommerce.csv /user/$USER/pertemuan4/

# 3. Memverifikasi keberadaan berkas di klaster HDFS
!hdfs dfs -ls -h /user/$USER/pertemuan4

In [ ]:
import os

# Membaca CSV LANGSUNG dari klaster HDFS ke dalam Spark DataFrame
username = os.environ.get("USER", "kaky")
hdfs_path = f"hdfs://localhost:9000/user/{username}/pertemuan4/data_transaksi_ecommerce.csv"

print(f"Membaca dataset langsung dari URL: {hdfs_path}")
df_hdfs = spark.read.csv(hdfs_path, header=True, inferSchema=True)

print("Jumlah baris yang dimuat dari HDFS:", df_hdfs.count())
df_hdfs.show(5)

Keberhasilan pembacaan data di atas membuktikan bahwa **arsitektur modern Big Data telah aktif di laptop Arch Linux Anda**:
* **Lapisan Penyimpanan (*Storage Layer*)**: Dikelola secara terdistribusi oleh **HDFS**.
* **Lapisan Pemrosesan (*Compute Layer*)**: Dijalankan secara cepat dalam memori RAM oleh **Apache Spark**.

---

### Menutup `SparkSession`
Sebagai kebiasaan baik dalam rekayasa data, selalu tutup `SparkSession` setelah sesi analisis selesai untuk membebaskan alokasi thread CPU dan memori JVM:

In [ ]:
spark.stop()
print("SparkSession berhasil ditutup. Alokasi resource CPU & RAM telah dibebaskan.")

---
## 7. Latihan Mandiri

Jalankan cell di bawah ini untuk membuat sesi `SparkSession` baru khusus latihan, lalu kerjakan 4 soal latihan di bawahnya.

In [ ]:
# Inisialisasi SparkSession baru khusus Latihan Mandiri
spark = SparkSession.builder.appName("Latihan4-ArchLinux").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Memuat data dan menambahkan kolom total_pendapatan
df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True)
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
print(f"Lingkungan Latihan Siap. Total Baris: {df.count()}")

**Soal 1.** Tampilkan hanya kolom `order_id`, `kategori`, dan `total_pendapatan` untuk seluruh transaksi yang menggunakan metode pembayaran `"E-Wallet"`.

In [ ]:
# Tuliskan kode jawaban Soal 1 di bawah ini:


**Soal 2.** Hitung total pendapatan kumulatif **per kategori produk** (bukan per kota), lalu urutkan dari kategori yang menghasilkan pendapatan tertinggi.

In [ ]:
# Tuliskan kode jawaban Soal 2 di bawah ini:


**Soal 3.** Tampilkan frekuensi jumlah transaksi untuk masing-masing `metode_pembayaran` **tanpa menggunakan fungsi `agg()`** (petunjuk: manfaatkan metode `count()` langsung setelah `groupBy()`).

In [ ]:
# Tuliskan kode jawaban Soal 3 di bawah ini:


**Soal 4 (Refleksi Konsep).** Dalam 2-3 kalimat: apa yang dimaksud dengan mekanisme ***lazy evaluation*** di PySpark, dan mengapa hal tersebut memberikan keuntungan efisiensi yang sangat signifikan saat memproses data berskala besar?

*(Tuliskan jawaban refleksi Anda pada sel ini)*

---
## 8. TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> 📅 **Tenggat Pengumpulan:** Sebelum Pertemuan 5 dimulai (via sistem ELITA).  
> 👤 **Sifat Tugas:** Individu.

### Skenario Kasus Bisnis
Tim rekayasa data (*Data Engineering*) platform e-commerce meminta agar seluruh pipeline analitik yang sebelumnya menggunakan Pandas **sepenuhnya dimigrasikan ke PySpark**. Langkah ini krusial karena volume data transaksi bulanan perusahaan mulai melonjak pesat sehingga pendekatan konvensional *in-memory* Pandas sudah tidak lagi mampu menangani skala data tanpa risiko *out-of-memory crash*.

Sebagai analis data, Anda ditugaskan membuktikan keandalan pipeline baru ini dengan memproses transaksi bulan September 2026 secara langsung dari klaster **HDFS**.

---

### Persiapan Dataset Tugas Mandiri
Jalankan cell di bawah ini untuk membangkitkan 1.000 transaksi sintetis bulan September 2026 dan mengunggahnya secara otomatis ke HDFS:

In [ ]:
import os
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data_tugas = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}

df_tugas4 = pd.DataFrame(data_tugas)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset lokal berhasil dibuat: {df_tugas4.shape[0]} baris.")

# Mengunggah ke HDFS menggunakan variabel dinamis $USER
!hdfs dfs -mkdir -p /user/$USER/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/$USER/tugas4/
print(f"Dataset berhasil diunggah ke HDFS: /user/{os.environ.get('USER', 'kaky')}/tugas4/transaksi_september_2026.csv")

### Instruksi Pengerjaan Tugas Mandiri

Buat notebook baru bernama **`Tugas4_[NIM]_[Nama Lengkap].ipynb`** di dalam folder `pertemuan-04/`. Buat `SparkSession` baru, lalu selesaikan bagian **A sampai E** berikut:
*(Wajib menggunakan PySpark murni — dilarang menggunakan Pandas untuk proses analisis dan manipulasi data!)*

---

#### A. Membaca dan Eksplorasi Awal *(Bobot 15%)*
* Baca berkas `transaksi_september_2026.csv` **langsung dari HDFS** (`hdfs://localhost:9000/user/$USER/tugas4/...`).
* Tampilkan skema data (`printSchema()`), total baris (`count()`), dan 10 baris pertama (`show(10)`).

#### B. Menangani Nilai Kosong (*Missing Value*) *(Bobot 15%)*
* Kolom `rating` memiliki nilai kosong (`null`/`NaN`).
* Hitung dan tampilkan berapa banyak nilai kosong pada kolom `rating`.
* Lakukan penanganan menggunakan salah satu metode: `df.na.fill()` (misal diisi nilai rata-rata/median) atau `df.na.drop()`. Berikan alasan logis atas metode yang Anda pilih dalam markdown cell.

#### C. Transformasi Kolom *(Bobot 20%)*
* Tambahkan kolom `total_pendapatan` hasil perkalian `unit_terjual * harga_satuan`.
* Tambahkan kolom kategori bersyarat `tier_transaksi`:
  - Bernilai `"Besar"` jika `total_pendapatan > 500000`.
  - Bernilai `"Kecil"` jika sebaliknya.
  *(Petunjuk: gunakan `when()` dan `otherwise()` dari `pyspark.sql.functions`).*

#### D. Analisis Data dengan GroupBy *(Bobot 30%)*
Jawab pertanyaan bisnis berikut menggunakan kueri PySpark:
1. Kategori produk apa yang menyumbang `total_pendapatan` tertinggi secara keseluruhan?
2. Kota mana yang mencatatkan frekuensi transaksi bertipe **tier "Besar"** terbanyak?
3. Berapakah rata-rata `rating` kepuasan pelanggan untuk masing-masing `metode_pembayaran`?

#### E. Menyimpan Hasil Akhir ke HDFS *(Bobot 20%)*
* Simpan DataFrame hasil transformasi (yang telah dilengkapi kolom `total_pendapatan` dan `tier_transaksi`) kembali ke dalam HDFS dalam format CSV baru di direktori `/user/$USER/tugas4/hasil_olahan/`.
* Verifikasi keberadaan berkas di HDFS menggunakan `!hdfs dfs -ls -h /user/$USER/tugas4/hasil_olahan/`.
* **Pertanyaan Konseptual:** Jelaskan pada markdown cell mengapa Spark menghasilkan beberapa berkas partisi bertuliskan `part-00000...` di direktori HDFS alih-alih satu berkas tunggal seperti pada Pandas.

---

### Ketentuan Pengumpulan
1. File yang dikumpulkan: **`Tugas4_250506070_Sholahuddin Ahmad.ipynb`** (sesuaikan NIM & Nama) via ELITA.
2. Pastikan cluster Hadoop aktif saat pengerjaan dan semua cell telah dijalankan secara berurutan (**Run All**) dengan output yang lengkap dan terbaca.
3. Seluruh instruksi komputasi dan analisis wajib menggunakan fungsi resmi PySpark.